# Day 2 · Notebook 02 — Making the model smaller and faster: Quantization & Pruning

We take the FP32 model from Notebook 01 and measure what each optimisation technique really buys us on a
CPU — **size on disk, latency (batch = 1) and accuracy**, always on the same 2 000 test images.

| # | Technique | What changes |
|---|---|---|
| 0 | FP32 baseline | nothing |
| a | Dynamic quantization | `Linear` weights → INT8, activations quantized on the fly |
| b | Static post-training quantization (PTQ) | weights **and** activations INT8, ranges from calibration data |
| c | FP16 / BF16 | 16-bit floating point |
| d | L1 unstructured pruning 30 % / 50 % | smallest weights set to zero |
| e | Quantization-aware training (QAT) | fine-tune with simulated INT8 rounding, then convert |

Runtime: ~6–8 min on a laptop CPU (QAT is the slowest part). Requires `../models/mobilenetv2_cifar10_fp32.pth`
(Notebook 01 or `scripts/train_and_export.py`).

## 0. Setup — and a word about `torch.ao.quantization` deprecation

In PyTorch 2.9 every use of `torch.ao.quantization` prints a `DeprecationWarning`:

> *torch.ao.quantization is deprecated and will be removed in 2.10 … migrate to torchao*

What this means for you:

* The **eager-mode** API used here (`quantize_dynamic`, `prepare`/`convert`, `prepare_qat`) still works in 2.9
  and is the clearest way to *learn* the concepts (observers, fusion, calibration, fake-quant).
* New quantization development happens in the separate library **`torchao`** (`pip install torchao`):
  `quantize_(model, Int8DynamicActivationInt8WeightConfig())` for eager-style quantization, and the
  **PT2E** flow (`torch.export` → `prepare_pt2e` → calibrate → `convert_pt2e`) which replaces FX graph mode.
* The *concepts* (scale, zero-point, per-channel, calibration, QAT) are identical — only the entry points move.
* For **deployment** in this course we don't depend on either: we quantize the ONNX model with
  ONNX Runtime in Notebook 03.

We silence only that specific warning so the output stays readable (plus two harmless warnings explained in
section 3b).

In [ ]:
QUICK_RUN = False      # True -> fewer test images / tiny QAT for a smoke test

import warnings
warnings.filterwarnings("ignore", message=r".*torch\.ao\.quantization is deprecated.*")
# two harmless warnings of the eager quantization flow (explained in section 3b):
warnings.filterwarnings("ignore", message=r".*must run observer before calling calculate_qparams.*")
warnings.filterwarnings("ignore", message=r".*reduce_range of false may have accuracy issues.*")

import copy, gzip, io, json, os, tempfile, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune
import torch.ao.quantization as tq
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.models import mobilenet_v2
from torchvision.models.quantization import mobilenet_v2 as quantizable_mobilenet_v2

torch.manual_seed(42)
DATA_DIR, MODELS_DIR = Path("..") / "data", Path("..") / "models"
OUT_DIR = Path("outputs"); OUT_DIR.mkdir(exist_ok=True)
IMG_SIZE, SEED = 96, 42
TEST_SUBSET = 500 if QUICK_RUN else 2000
CALIB_SIZE = 100 if QUICK_RUN else 500
BENCH_THREADS = 4          # fixed thread count for every latency measurement (fair comparison)
LABELS = json.loads((MODELS_DIR / "labels.json").read_text())
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

# --- quantized-kernel backend ------------------------------------------------------------
# x86 (FBGEMM + oneDNN) is the recommended server backend, qnnpack is for ARM/mobile.
# Some wheels ship only a subset -> pick the first one this build actually supports.
print("supported quantized engines:", torch.backends.quantized.supported_engines)
ENGINE = next(e for e in ["x86", "fbgemm", "onednn", "qnnpack"]
              if e in torch.backends.quantized.supported_engines)
torch.backends.quantized.engine = ENGINE
print("using engine:", ENGINE, "| torch", torch.__version__)

**Backend note.** The quantized kernels are provided by a backend library:

| Backend | Target | Notes |
|---|---|---|
| `x86` | Intel/AMD servers & laptops | default on Linux x86 wheels; combines FBGEMM + oneDNN |
| `fbgemm` | x86 with AVX2/AVX-512 | Facebook's GEMM library, the classic server choice |
| `onednn` | x86 (Intel oneDNN) | used when FBGEMM is not compiled in |
| `qnnpack` | ARM (phones, Raspberry Pi, Apple M-series) | mobile backend |

The **qconfig must match the backend you will run on** (e.g. `fbgemm` uses `reduce_range=True` to avoid
overflow in its 16-bit accumulation path, `onednn`/`qnnpack` do not). On the Windows CPU wheel of torch 2.9.1
only `onednn` is available — the code above picks whatever the machine supports, so the same notebook
works on Linux (`x86`) and Mac/ARM (`qnnpack`).

## 1. Data and the FP32 baseline

In [ ]:
test_tf = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(),
                              transforms.Normalize(MEAN, STD)])
test_full = datasets.CIFAR10(DATA_DIR, train=False, download=True, transform=test_tf)
train_plain = datasets.CIFAR10(DATA_DIR, train=True, download=True, transform=test_tf)  # for calibration

def fixed_subset(n_total, n, seed):
    g = torch.Generator().manual_seed(seed)
    return torch.randperm(n_total, generator=g)[:n].tolist()

test_ds = Subset(test_full, fixed_subset(len(test_full), TEST_SUBSET, SEED))       # same as Notebook 01
calib_ds = Subset(train_plain, fixed_subset(len(train_plain), CALIB_SIZE, SEED + 1)) # TRAIN images only!
test_loader = DataLoader(test_ds, batch_size=200, shuffle=False)
calib_loader = DataLoader(calib_ds, batch_size=50, shuffle=False)

def load_fp32():
    m = mobilenet_v2(weights=None)
    m.classifier[1] = nn.Linear(1280, 10)
    m.load_state_dict(torch.load(MODELS_DIR / "mobilenetv2_cifar10_fp32.pth", map_location="cpu",
                                 weights_only=True))
    return m.eval()

fp32_model = load_fp32()
print(f"test images: {len(test_ds)} | calibration images: {len(calib_ds)} (from the training set)")

Calibration images come from the **training** set — never tune anything (not even quantization ranges) on
the test set.

### Measurement helpers

* **Size:** serialise the `state_dict` to a file and read its size — what you would ship.
* **Latency:** batch = 1 (an API serving one request), `torch.inference_mode()`, 10 warm-up runs (first calls
  allocate memory and pick kernels), then the **median** of N runs with a fixed number of threads. The median
  is robust to hiccups from other processes. *Always* benchmark on the target hardware.
* **Accuracy:** top-1 on the fixed test subset.

In [ ]:
def size_mb(model):
    with tempfile.NamedTemporaryFile(suffix=".pt", delete=False) as f:
        path = f.name
    torch.save(model.state_dict(), path)
    s = os.path.getsize(path) / 1e6
    os.remove(path)
    return s

def gzip_mb(model):
    buf = io.BytesIO(); torch.save(model.state_dict(), buf)
    return len(gzip.compress(buf.getvalue(), compresslevel=6)) / 1e6

@torch.inference_mode()
def latency_ms(model, n=100, warmup=10, dtype=torch.float32):
    old = torch.get_num_threads(); torch.set_num_threads(BENCH_THREADS)
    x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, dtype=dtype)
    model.eval()
    for _ in range(warmup):
        model(x)
    times = []
    for _ in range(n):
        t = time.perf_counter(); model(x); times.append((time.perf_counter() - t) * 1000)
    torch.set_num_threads(old)
    return float(np.median(times))

@torch.inference_mode()
def accuracy(model, loader=None, dtype=torch.float32):
    model.eval(); correct = total = 0
    for x, y in (loader or test_loader):
        correct += (model(x.to(dtype)).argmax(1) == y).sum().item(); total += y.numel()
    return correct / total

results = []
def record(name, model, size=None, lat=None, acc=None, note=""):
    row = {"technique": name,
           "size_mb": round(size if size is not None else size_mb(model), 2),
           "latency_ms": round(lat if lat is not None else latency_ms(model), 2),
           "accuracy": round(acc if acc is not None else accuracy(model), 4),
           "note": note}
    results.append(row); print(row); return row

base = record("FP32 baseline", fp32_model, note="reference")

This is our reference point: ~9 MB, a few ms to a few tens of ms per image on a CPU (depends heavily on the
machine), and the accuracy from Notebook 01.

## 2. Quantization in 3 minutes — scale & zero-point by hand

Affine (asymmetric) quantization maps a real value $x$ to an integer $q$:

$$q = \text{clamp}\big(\text{round}(x / s) + z,\; q_{min},\; q_{max}\big) \qquad \hat{x} = s\,(q - z)$$

with scale $s = \dfrac{x_{max} - x_{min}}{q_{max} - q_{min}}$ and zero-point $z = q_{min} - \text{round}(x_{min}/s)$.
The zero-point guarantees that the real value **0.0 is represented exactly** (important for zero padding and ReLU).

In [ ]:
x = torch.tensor([-1.8, -0.5, 0.0, 0.3, 1.1, 2.6])
qmin, qmax = 0, 255                                  # quint8
s = (x.max() - x.min()) / (qmax - qmin)
z = int(qmin - torch.round(x.min() / s))
q_manual = torch.clamp(torch.round(x / s) + z, qmin, qmax)
x_hat = s * (q_manual - z)

q_torch = torch.quantize_per_tensor(x, scale=float(s), zero_point=z, dtype=torch.quint8)
print(f"scale={s:.5f}  zero_point={z}")
print("x        :", x.tolist())
print("q (ours) :", q_manual.int().tolist())
print("q (torch):", q_torch.int_repr().tolist())
print("x_hat    :", [round(v, 4) for v in x_hat.tolist()])
print("max abs error:", float((x - x_hat).abs().max()), "<= s/2 =", float(s / 2))

The round-trip error is at most $s/2$. So: **the wider the range, the bigger the error.** That is why a single
outlier value hurts, and why *per-channel* scales help for weights.

### Per-tensor vs per-channel on a real layer

In [ ]:
w = fp32_model.features[1].conv[0][0].weight.detach()     # depthwise 3x3 conv, 32 output channels
per_ch_absmax = w.abs().amax(dim=(1, 2, 3))
print(f"weight shape {tuple(w.shape)} | per-channel |w|max ranges from "
      f"{per_ch_absmax.min():.3f} to {per_ch_absmax.max():.3f}")

# symmetric INT8 quantization of the weights
s_t = float(w.abs().max() / 127)
w_pt = torch.quantize_per_tensor(w, s_t, 0, torch.qint8).dequantize()
scales = per_ch_absmax / 127
w_pc = torch.quantize_per_channel(w, scales, torch.zeros(32, dtype=torch.long), 0, torch.qint8).dequantize()

err = lambda a: float(((a - w) ** 2).mean().sqrt())
print(f"RMS error per-tensor : {err(w_pt):.5f}")
print(f"RMS error per-channel: {err(w_pc):.5f}   (x{err(w_pt)/err(w_pc):.1f} smaller)")

Channels with small weights get crushed into a handful of integer levels when they share one scale with a
large-valued channel. Per-channel quantization gives each output channel its own scale — this is the default
for conv/linear **weights** in modern toolchains (PyTorch `x86`/`fbgemm`/`onednn` qconfigs, ONNX Runtime
`per_channel=True`). Activations stay per-tensor (a per-channel activation scale would complicate the integer
matmul).

## 3a. Dynamic quantization

* Weights converted to INT8 **ahead of time**.
* Activations quantized **on the fly** at each call (scale computed from the actual min/max) → no calibration.
* One line of code; works for `nn.Linear` and `nn.LSTM`/`nn.GRU`.
* Sweet spot: models dominated by large matmuls — **Transformers (BERT), LSTMs**.

In [ ]:
dyn_model = tq.quantize_dynamic(copy.deepcopy(fp32_model), {nn.Linear}, dtype=torch.qint8)
print(dyn_model.classifier)
n_linear = sum(p.numel() for m in fp32_model.modules() if isinstance(m, nn.Linear) for p in m.parameters())
n_all = sum(p.numel() for p in fp32_model.parameters())
print(f"parameters inside nn.Linear layers: {n_linear:,} of {n_all:,} ({100*n_linear/n_all:.1f} %)")
record("Dynamic INT8 (Linear only)", dyn_model, note="no calibration; only the head is quantized")

**Almost nothing changes — and that is the lesson.** MobileNetV2 is a *convolutional* network: only the final
`Linear(1280, 10)` (0.6 % of the weights) can be dynamically quantized; all the compute is in the convs.
Dynamic quantization is the right tool for a BERT model, not for a CNN. For CNNs we need **static**
quantization.

## 3b. Static post-training quantization (PTQ)

Steps (eager mode):

1. **Quantizable architecture** — torchvision ships `torchvision.models.quantization.mobilenet_v2`: same
   layers + `QuantStub`/`DeQuantStub` at the borders and `FloatFunctional` for the residual additions
   (a plain `+` cannot be quantized in eager mode). Same parameter names → our weights load directly.
2. **Fuse** `Conv + BatchNorm + ReLU` into one module: fewer memory round-trips and one requantization instead
   of three.
3. **Set qconfig** (observers for activations & weights, matching the backend).
4. **Prepare** — inserts observers that record activation ranges.
5. **Calibrate** — run a few hundred representative images through the model.
6. **Convert** — replace modules by INT8 kernels using the collected scales/zero-points.

In [ ]:
qmodel = quantizable_mobilenet_v2(weights=None, quantize=False)
qmodel.classifier[1] = nn.Linear(1280, 10)
print(qmodel.load_state_dict(fp32_model.state_dict()))      # same keys -> loads cleanly
qmodel.eval()

print("before fusion:", qmodel.features[1].conv)
qmodel.fuse_model(is_qat=False)                             # Conv+BN(+ReLU) -> ConvReLU2d / Conv2d
print("after fusion :", qmodel.features[1].conv)

After fusion the BatchNorm has been **folded into the convolution weights** ($w' = w\cdot\gamma/\sigma$,
$b' = (b-\mu)\gamma/\sigma + \beta$) and the ReLU6 is merged — the BN slot becomes `Identity`. The fused
float model still computes the same function (up to floating-point rounding).

Now set the qconfig, insert observers and calibrate:

In [ ]:
qmodel.qconfig = tq.get_default_qconfig(ENGINE)
print(qmodel.qconfig)
tq.prepare(qmodel, inplace=True)

t0 = time.time()
with torch.inference_mode():
    for x, _ in calib_loader:
        qmodel(x)                       # observers record min/max (histograms) of every activation
print(f"calibrated on {len(calib_ds)} images in {time.time()-t0:.1f}s")
print("example observer (first quant stub):", qmodel.quant.activation_post_process)

* Activations: `HistogramObserver` — builds a histogram and picks the range that minimises quantization
  error (clips rare outliers instead of letting them blow up the scale).
* Weights: `PerChannelMinMaxObserver` with symmetric INT8 → one scale per output channel.

About the two warnings we filtered in the setup cell:

* *"must run observer before calling calculate_qparams"* — every inverted-residual block of the quantizable
  model owns a `skip_add` (`FloatFunctional`) with an observer, but only blocks **with** a residual connection
  use it (7 blocks don't). Those unused observers never see data → harmless.
* *"oneDNN … reduce_range of false may have accuracy issues on CPU without VNNI"* — the oneDNN backend uses the
  full 8-bit activation range; on old CPUs without the VNNI instructions the 16-bit intermediate accumulation
  can saturate. Current Intel/AMD CPUs have VNNI. The `x86`/`fbgemm` qconfigs use `reduce_range=True`
  (7-bit activations) for exactly this reason.

Convert and measure:

In [ ]:
tq.convert(qmodel, inplace=True)
print(qmodel.features[1].conv)          # QuantizedConvReLU2d / QuantizedConv2d with scale & zero_point
static = record("Static PTQ INT8", qmodel, note=f"{len(calib_ds)} calibration images, engine={ENGINE}")
print(f"\nsize x{base['size_mb']/static['size_mb']:.1f} smaller | "
      f"speed-up x{base['latency_ms']/static['latency_ms']:.1f} | "
      f"accuracy change {100*(static['accuracy']-base['accuracy']):+.2f} pp")

**Expected:** ~3.5× smaller (INT8 = 1 byte instead of 4, plus a little overhead for scales), usually faster on
x86, and typically **< 1–2 percentage points** accuracy drop. MobileNet is actually one of the *harder*
architectures to quantize (depthwise convs have few weights per channel and wide ranges) — ResNets usually lose
even less.

⚠️ The quantized model runs **only on CPU** with the quantized backend it was prepared for — it will not
run on CUDA. For GPU deployment you would use TensorRT / FP16 instead.

## 3c. FP16 and BF16

| Format | Bits (sign/exponent/mantissa) | Range | Typical use |
|---|---|---|---|
| FP32 | 1 / 8 / 23 | ±3.4e38 | training & CPU inference |
| FP16 | 1 / 5 / 10 | ±65 504 | GPU inference (Tensor Cores) |
| BF16 | 1 / 8 / 7 | same as FP32 | training on TPU/modern GPUs, CPUs with AMX |

Halving the bits halves the size. Speed-up requires **hardware support**: Tensor Cores on GPUs, AMX/AVX-512-BF16 on
recent Xeons. On an ordinary laptop CPU PyTorch has to emulate these types — let's measure.

In [ ]:
fp16_model = copy.deepcopy(fp32_model).half()
fp16_size = size_mb(fp16_model)
# accuracy effect of *storing* weights in FP16 (compute in FP32): cast weights down and back up
fp16_roundtrip = copy.deepcopy(fp32_model).half().float()
fp16_acc = accuracy(fp16_roundtrip)
record("FP16 weights (storage)", fp16_model, size=fp16_size, lat=base["latency_ms"], acc=fp16_acc,
       note="size halves; latency = FP32 (weights up-cast); FP16 speed-ups need a GPU")

bf16_model = copy.deepcopy(fp32_model).to(torch.bfloat16)
bf16_lat = latency_ms(bf16_model, n=10, warmup=2, dtype=torch.bfloat16)
bf16_acc = accuracy(bf16_model, dtype=torch.bfloat16)
record("BF16 on CPU", bf16_model, lat=bf16_lat, acc=bf16_acc,
       note="no native BF16 units on this CPU -> slow emulation")

* Accuracy is essentially unchanged: 16-bit floats are plenty for inference.
* Size halves.
* On a CPU **without** native BF16/FP16 units, the 16-bit model is *much slower* — a reminder that
  "fewer bits" only means "faster" when the hardware has kernels for them. On an NVIDIA GPU, FP16 is often 2×+
  faster than FP32; on CPUs, **INT8** is the format with broad, fast kernel support.

## 3d. Pruning — unstructured L1 magnitude pruning

Idea: many weights are close to zero and contribute little. **Global L1 unstructured pruning** removes the
p % of weights with the smallest |w| across all conv/linear layers.

`torch.nn.utils.prune` implements pruning with a **mask**: the original weight is kept as `weight_orig`, a
`weight_mask` buffer is added, and `weight = weight_orig * weight_mask` is recomputed in a forward pre-hook.
`prune.remove()` makes it permanent (bakes the zeros into `weight`).

In [ ]:
def global_prune(model, amount, make_permanent=True):
    m = copy.deepcopy(model)
    params = [(mod, "weight") for mod in m.modules() if isinstance(mod, (nn.Conv2d, nn.Linear))]
    prune.global_unstructured(params, pruning_method=prune.L1Unstructured, amount=amount)
    if make_permanent:
        for mod, name in params:
            prune.remove(mod, name)      # make permanent: weight now contains real zeros
    return m

def sparsity(model):
    ws = [mod.weight for mod in model.modules() if isinstance(mod, (nn.Conv2d, nn.Linear))]
    zeros = sum(int((w == 0).sum()) for w in ws); total = sum(w.numel() for w in ws)
    return zeros / total

pruned = {}
for amount in [0.3, 0.5]:
    pm = global_prune(fp32_model, amount)
    pruned[amount] = pm
    print(f"pruned {int(amount*100)}%: sparsity = {sparsity(pm):.3f} | "
          f"raw size {size_mb(pm):.2f} MB | gzip size {gzip_mb(pm):.2f} MB "
          f"(FP32 gzip {gzip_mb(fp32_model):.2f} MB)")
    record(f"Pruned {int(amount*100)}% (L1 unstructured)", pm,
           note="zeros stored densely -> same size & speed")

Observe three things:

1. **Size on disk is unchanged** — a zero still occupies 4 bytes in a dense tensor. Only after *compression*
   (gzip) or a sparse storage format does the file shrink.
2. **Latency is unchanged** (differences are measurement noise) — dense conv/GEMM kernels multiply by zero just
   as fast as by any other number. Speed-ups need either *structured* pruning (remove whole channels/filters →
   genuinely smaller dense layers) or special sparse kernels / hardware (e.g. NVIDIA 2:4 sparsity).
3. **Accuracy drops** as sparsity grows, especially without re-training. In practice you prune *gradually* and
   fine-tune between steps ("prune → fine-tune → prune…"), which recovers most of the accuracy.

### Prune → fine-tune: recovering accuracy

Let's give the 50 % model one short fine-tuning pass (2 000 augmented training images). The pruning **masks
stay attached** during training (`weight = weight_orig * weight_mask` is recomputed at every forward), so pruned
weights stay exactly zero while the surviving weights adapt. Only afterwards do we call `prune.remove()`.

In [ ]:
SMALL_TRAIN = 256 if QUICK_RUN else 2000
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomCrop(IMG_SIZE, padding=IMG_SIZE // 12, padding_mode="reflect"),
    transforms.RandomHorizontalFlip(), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
train_aug = datasets.CIFAR10(DATA_DIR, train=True, download=True, transform=train_tf)
small_train_loader = DataLoader(Subset(train_aug, fixed_subset(len(train_aug), SMALL_TRAIN, SEED + 2)),
                                batch_size=32, shuffle=True, drop_last=True)

pm_ft = global_prune(fp32_model, 0.5, make_permanent=False)     # masks still attached
opt = torch.optim.AdamW(pm_ft.parameters(), lr=3e-4, weight_decay=1e-4)
crit = nn.CrossEntropyLoss()
pm_ft.train(); t0 = time.time()
for x, y in small_train_loader:
    loss = crit(pm_ft(x), y)
    opt.zero_grad(); loss.backward(); opt.step()
for mod in pm_ft.modules():
    if isinstance(mod, (nn.Conv2d, nn.Linear)):
        prune.remove(mod, "weight")
pm_ft.eval()
print(f"fine-tuned {len(small_train_loader)} steps in {time.time()-t0:.0f}s | sparsity still {sparsity(pm_ft):.3f}")
record("Pruned 50% + short fine-tune", pm_ft, note=f"1 pass over {SMALL_TRAIN} images, masks kept")

A single short pass recovers most of the lost accuracy while keeping 50 % of the weights at zero. Real
pipelines iterate: prune a bit more → fine-tune → repeat ("gradual magnitude pruning").

Where does the pruning hit? Per-layer sparsity for the 50 % model (before fine-tuning):

In [ ]:
pm = pruned[0.5]
names, sp = [], []
for name, mod in pm.named_modules():
    if isinstance(mod, (nn.Conv2d, nn.Linear)):
        names.append(name); sp.append(float((mod.weight == 0).float().mean()))
fig, ax = plt.subplots(figsize=(13, 3.2))
colors = ["tab:orange" if isinstance(dict(pm.named_modules())[n], nn.Conv2d) and
          dict(pm.named_modules())[n].groups > 1 else "tab:blue" for n in names]
ax.bar(range(len(sp)), sp, color=colors)
ax.set_xlabel("layer index (orange = depthwise conv)"); ax.set_ylabel("sparsity"); ax.set_ylim(0, 1)
ax.set_title("Per-layer sparsity after 50 % global L1 pruning"); plt.show()

Global pruning does **not** prune every layer equally: layers with many small weights (large 1×1 convs) are
pruned heavily, the few, important depthwise weights much less. Uniform per-layer pruning (`prune.l1_unstructured`
on each layer) would hurt accuracy more.

## 3e. Quantization-aware training (QAT) — short demo

PTQ quantizes a model that never "saw" rounding. QAT inserts **fake-quantization** modules
(quantize → dequantize in float) during fine-tuning, so the network learns weights that are robust to INT8
rounding. Gradients pass through the rounding with the *straight-through estimator* (STE).

Usually recovers most of the PTQ accuracy gap; costs a (short) extra training run. We run 1 epoch on a small
subset with a low learning rate — just enough to show the workflow.

In [ ]:
qat_loader = small_train_loader          # the same 2 000 augmented training images as in 3d

qat = quantizable_mobilenet_v2(weights=None, quantize=False)
qat.classifier[1] = nn.Linear(1280, 10)
qat.load_state_dict(fp32_model.state_dict())
qat.train()
qat.fuse_model(is_qat=True)                              # fuse but keep BN trainable (ConvBnReLU)
qat.qconfig = tq.get_default_qat_qconfig(ENGINE)
tq.prepare_qat(qat, inplace=True)                        # inserts FakeQuantize modules

opt = torch.optim.SGD(qat.parameters(), lr=1e-4, momentum=0.9)
crit = nn.CrossEntropyLoss()
t0 = time.time()
n_steps = len(qat_loader)
for step, (x, y) in enumerate(qat_loader):
    if step == n_steps // 2:
        qat.apply(tq.disable_observer)                   # freeze quantization ranges
        qat.apply(torch.ao.nn.intrinsic.qat.freeze_bn_stats)  # freeze BN running stats
    loss = crit(qat(x), y)
    opt.zero_grad(); loss.backward(); opt.step()
print(f"QAT: {n_steps} steps in {time.time()-t0:.0f}s, last loss {loss.item():.3f}")

qat.eval()
qat_int8 = tq.convert(qat)
record("QAT INT8 (1 short epoch)", qat_int8, note=f"{SMALL_TRAIN} images, lr 1e-4")

Freezing the observers and BatchNorm statistics halfway is the recommended recipe: the quantization grid
stops moving and the network fine-tunes against a fixed grid. With only 62 steps on 2 000 images, QAT lands
**within a few tenths of a percentage point of PTQ** (slightly above or below, depending on the run) — our PTQ
drop is already tiny, so there is little to recover. With a few full epochs, QAT usually closes most of the gap
to FP32 for models where PTQ loses more. Rule of thumb: try PTQ
first; use QAT only if the PTQ accuracy drop is unacceptable.

## 4. What difference does it make? — Summary

In [ ]:
df = pd.DataFrame(results)
df["size_vs_fp32"] = (df["size_mb"] / base["size_mb"]).round(2)
df["speedup_vs_fp32"] = (base["latency_ms"] / df["latency_ms"]).round(2)
df["acc_delta_pp"] = (100 * (df["accuracy"] - base["accuracy"])).round(2)
df.to_csv(OUT_DIR / "day2_torch_optimization_results.csv", index=False)
pd.set_option("display.max_colwidth", 80)
display(df[["technique", "size_mb", "latency_ms", "accuracy", "size_vs_fp32", "speedup_vs_fp32",
            "acc_delta_pp", "note"]])

`size_vs_fp32` < 1 means smaller, `speedup_vs_fp32` > 1 means faster, `acc_delta_pp` is the accuracy change in
percentage points. The same numbers as bar charts:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
colors = plt.cm.tab10(np.arange(len(df)))
for ax, col, title in zip(axes, ["size_mb", "latency_ms", "accuracy"],
                          ["Size on disk [MB] (lower is better)",
                           f"Latency, batch=1, {BENCH_THREADS} threads [ms] (log scale)",
                           "Top-1 accuracy (higher is better)"]):
    ax.barh(df["technique"], df[col], color=colors)
    ax.set_title(title); ax.invert_yaxis(); ax.grid(axis="x", alpha=.3)
    for i, v in enumerate(df[col]):
        ax.text(v, i, f" {v:.3g}", va="center", fontsize=8)
axes[1].set_xscale("log")
axes[2].set_xlim(max(0, df["accuracy"].min() - 0.1), 1.0)
for ax in axes[1:]:
    ax.set_yticklabels([])
plt.tight_layout(); plt.show()

### Discussion

* **Static INT8** is the big win for CNNs on CPU: ~3.5× smaller and faster with a small accuracy cost.
* **Dynamic quantization** does nothing useful for a conv net (but is great for Transformers/LSTMs).
* **FP16/BF16** halve the size but only speed things up on hardware with native support (GPUs, AMX CPUs).
* **Unstructured pruning** reduces *compressed* size, not dense latency; accuracy suffers without fine-tuning
  and largely recovers with it.
* **QAT** trades extra training for accuracy closest to FP32 at INT8 cost.
* Latency numbers are **machine-specific** (and noisy on a shared laptop) — the ratios are what to discuss.

Note that PyTorch eager INT8 models are awkward to *deploy* (they need the right backend + the same Python
code). In **Notebook 03** we export to ONNX and let ONNX Runtime do both graph optimisation and INT8
quantization — that is what we will serve on Day 3.